# NB - YOLO + Vision Transformer

Pipeline: YOLO crop vùng bệnh rồi train ViT trên ảnh crop.

In [ ]:
# 1. INSTALL
!pip install -q ultralytics

In [ ]:
# 2. IMPORT

import os
import json
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import confusion_matrix, classification_report

from ultralytics import YOLO

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 3. CONFIG

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_PATH = "/kaggle/input/datasets/kimlonkkk/data10k-v2/split_dataset_backup_10k_v2"

YOLO_MODEL_PATH = "/kaggle/working/best_yolo_corn.pt"
# Nếu chưa copy best.pt thì đổi thành:
# YOLO_MODEL_PATH = "/kaggle/working/corn_yolo/weights/best.pt"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 40

MODEL_NAME = "YOLO_ViT"
OUT_DIR = f"/kaggle/working/{MODEL_NAME}_results"
CROP_DATASET_DIR = "/kaggle/working/yolo_crop_classification_dataset"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CROP_DATASET_DIR, exist_ok=True)

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# 4. CHECK PATHS

print("DATASET_PATH exists:", os.path.exists(DATASET_PATH))
print("YOLO_MODEL_PATH exists:", os.path.exists(YOLO_MODEL_PATH))
print("Dataset folders:", os.listdir(DATASET_PATH))

In [ ]:
# 5. LOAD YOLO MODEL

yolo_model = YOLO(YOLO_MODEL_PATH)
print("YOLO classes:", yolo_model.names)

In [ ]:
# 6. CROP FUNCTION

def crop_with_yolo(image_path, yolo_model, conf_thres=0.25, img_size=224):
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    results = yolo_model.predict(
        source=str(image_path),
        conf=conf_thres,
        verbose=False
    )

    boxes = results[0].boxes

    if boxes is not None and len(boxes) > 0:
        confs = boxes.conf.cpu().numpy()
        best_idx = int(np.argmax(confs))
        xyxy = boxes.xyxy[best_idx].cpu().numpy()
        x1, y1, x2, y2 = xyxy

        pad_x = int((x2 - x1) * 0.15)
        pad_y = int((y2 - y1) * 0.15)

        x1 = max(0, int(x1) - pad_x)
        y1 = max(0, int(y1) - pad_y)
        x2 = min(w, int(x2) + pad_x)
        y2 = min(h, int(y2) + pad_y)

        if x2 > x1 and y2 > y1:
            image = image.crop((x1, y1, x2, y2))

    image = image.resize((img_size, img_size))
    return image

In [ ]:
# 7. CREATE CROPPED DATASET

splits = ["train", "val", "test"]
image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

crop_stats = {"total": 0, "saved": 0, "errors": 0}

for split in splits:
    split_dir = Path(DATASET_PATH) / split

    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name
        out_class_dir = Path(CROP_DATASET_DIR) / split / class_name
        out_class_dir.mkdir(parents=True, exist_ok=True)

        image_paths = [p for p in class_dir.iterdir() if p.suffix.lower() in image_exts]
        print(f"Processing {split}/{class_name}: {len(image_paths)} images")

        for img_path in image_paths:
            crop_stats["total"] += 1

            try:
                cropped = crop_with_yolo(
                    image_path=img_path,
                    yolo_model=yolo_model,
                    conf_thres=0.25,
                    img_size=IMG_SIZE
                )

                out_path = out_class_dir / img_path.name
                cropped.save(out_path)
                crop_stats["saved"] += 1

            except Exception as e:
                crop_stats["errors"] += 1
                print("Error:", img_path, e)

print("Crop stats:", crop_stats)
print("Cropped dataset saved at:", CROP_DATASET_DIR)

In [ ]:
# 8. VISUALIZE CROPPED IMAGES

import glob

sample_images = glob.glob(CROP_DATASET_DIR + "/train/*/*")[:9]

plt.figure(figsize=(12, 12))

for i, img_path in enumerate(sample_images):
    img = Image.open(img_path).convert("RGB")

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(Path(img_path).parent.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 9. LOAD CROPPED DATASET

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CROP_DATASET_DIR, "train"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CROP_DATASET_DIR, "val"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(CROP_DATASET_DIR, "test"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)
print("Num classes:", NUM_CLASSES)

In [ ]:
# 10. NORMALIZE + PREFETCH

normalization_layer = layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
# 11. DATA AUGMENTATION NHẸ

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomZoom(0.03),
        layers.RandomTranslation(0.02, 0.02),
    ],
    name="data_augmentation"
)

In [ ]:
# 12. PATCH LAYER

class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        patch_dims = tf.shape(patches)[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config

In [ ]:
# 13. PATCH ENCODER

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim
        })
        return config

In [ ]:
# 14. MLP HELPER

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
# 15. BUILD ViT MODEL

PATCH_SIZE = 32
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2

PROJECTION_DIM = 32
NUM_HEADS = 2
TRANSFORMER_LAYERS = 4

TRANSFORMER_UNITS = [PROJECTION_DIM * 2, PROJECTION_DIM]
MLP_HEAD_UNITS = [128]

def build_vit_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)

    patches = Patches(PATCH_SIZE, name="patches")(x)

    encoded_patches = PatchEncoder(
        NUM_PATCHES,
        PROJECTION_DIM,
        name="patch_encoder"
    )(patches)

    for i in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6, name=f"ln_1_block_{i+1}")(encoded_patches)

        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=PROJECTION_DIM,
            dropout=0.1,
            name=f"mha_block_{i+1}"
        )(x1, x1)

        x2 = layers.Add(name=f"skip_1_block_{i+1}")([attention_output, encoded_patches])

        x3 = layers.LayerNormalization(epsilon=1e-6, name=f"ln_2_block_{i+1}")(x2)

        x3 = mlp(x3, hidden_units=TRANSFORMER_UNITS, dropout_rate=0.1)

        encoded_patches = layers.Add(name=f"skip_2_block_{i+1}")([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6, name="encoder_norm")(encoded_patches)
    representation = layers.Flatten(name="flatten")(representation)
    representation = layers.Dropout(0.3, name="head_dropout")(representation)

    features = mlp(representation, hidden_units=MLP_HEAD_UNITS, dropout_rate=0.3)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classification_head")(features)

    model = Model(inputs=inputs, outputs=outputs, name=MODEL_NAME)
    return model

model = build_vit_model()
model.summary()

In [ ]:
# 16. COMPILE

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# 17. CALLBACKS

checkpoint = ModelCheckpoint(
    filepath=f"{OUT_DIR}/{MODEL_NAME}_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

earlystop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

callbacks = [checkpoint, earlystop, reduce_lr]

In [ ]:
# 18. TRAIN

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# 19. SAVE HISTORY

history_df = pd.DataFrame(history.history)
history_path = f"{OUT_DIR}/history.csv"
history_df.to_csv(history_path, index=False)

print("Saved:", history_path)
history_df.head()

In [ ]:
# 20. LEARNING CURVES

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="train_accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.title("YOLO + ViT Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUT_DIR}/accuracy_curve.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("YOLO + ViT Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig(f"{OUT_DIR}/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 21. EVALUATE TEST SET

test_loss, test_acc = model.evaluate(test_ds)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

In [ ]:
# 22. PREDICT TEST SET

y_true = []

for _, y_batch in test_ds:
    y_true.extend(y_batch.numpy())

y_true = np.array(y_true)

pred_probs = model.predict(test_ds)
y_pred = np.argmax(pred_probs, axis=1)

print("y_true:", y_true.shape)
print("y_pred:", y_pred.shape)

In [ ]:
# 23. CONFUSION MATRIX

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("YOLO + ViT Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

cm_path = f"{OUT_DIR}/confusion_matrix.png"
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved:", cm_path)

In [ ]:
# 24. CLASSIFICATION REPORT

report_text = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES
)

print(report_text)

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()
report_path = f"{OUT_DIR}/classification_report.csv"
report_df.to_csv(report_path)

print("Saved:", report_path)
report_df

In [ ]:
# 25. SAVE METRICS

metrics = {
    "model_name": MODEL_NAME,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "classes": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "yolo_model_path": YOLO_MODEL_PATH,
    "crop_dataset_dir": CROP_DATASET_DIR,
    "patch_size": PATCH_SIZE,
    "num_patches": NUM_PATCHES,
    "projection_dim": PROJECTION_DIM,
    "num_heads": NUM_HEADS,
    "transformer_layers": TRANSFORMER_LAYERS
}

metrics_path = f"{OUT_DIR}/metrics.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4, ensure_ascii=False)

print("Saved:", metrics_path)

In [ ]:
# 26. ZIP RESULTS

zip_path = f"/kaggle/working/{MODEL_NAME}_results.zip"

shutil.make_archive(
    zip_path.replace(".zip", ""),
    "zip",
    OUT_DIR
)

print("Saved:", zip_path)

In [ ]:
# 27. OPTIONAL - TEST 1 IMAGE FROM COMPUTER

import io
from PIL import Image
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(
    accept="image/*",
    multiple=False
)

display(uploader)

In [ ]:
# Run this cell after choosing an image above.

uploaded = uploader.value[0]

img = Image.open(io.BytesIO(uploaded["content"])).convert("RGB")

temp_path = "/kaggle/working/temp_upload_image.jpg"
img.save(temp_path)

cropped = crop_with_yolo(
    image_path=temp_path,
    yolo_model=yolo_model,
    conf_thres=0.25,
    img_size=IMG_SIZE
)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cropped)
plt.title("YOLO crop")
plt.axis("off")

plt.show()

x = np.array(cropped).astype("float32") / 255.0
x = np.expand_dims(x, axis=0)

pred = model.predict(x)[0]

pred_idx = int(np.argmax(pred))
confidence = float(pred[pred_idx])

print("Prediction:", CLASS_NAMES[pred_idx])
print("Confidence:", f"{confidence:.4f}")

print("\nAll probabilities:")
for cls, prob in zip(CLASS_NAMES, pred):
    print(f"{cls:15s}: {prob:.4f}")